## 5 · Self-study: Essentia mood classifiers
Not covered in the tutorial – work through it at home. It needs its **own environment** and runs in its **own kernel**.

### 5.1 · Which mood models?
| Model family | Output | Available via pip? |
|---|---|---|
| SVM models on the lecture slide (Laurier et al. 2009, AcousticBrainz / "Gaia" extractor) | happy / sad / … class probabilities | **no** – only in self-compiled Essentia with Gaia |
| **TensorFlow classifier heads** on audio embeddings: `mood_happy`, `mood_sad`, `mood_relaxed`, `mood_aggressive`, `mood_party`, `mood_acoustic`, `mood_electronic` | per mood: probability 0 … 1 | **yes** (`essentia-tensorflow`) |
| MTG-Jamendo mood/theme (56 tags) | 56 probabilities | yes |
| Arousal/valence regression (DEAM, emoMusic) | two continuous values | yes |

→ We use the **seven binary mood classifiers** on **Discogs-EffNet** embeddings: one **confidence score per mood**, no fixed tag.
They cover the same seven moods as the old SVM models (Laurier et al. 2009), but they are newer models
(Alonso-Jiménez et al. 2020, 2022) – their accuracies are *not* the ones on the slide.

How they work: a large network (**Discogs-EffNet**, trained to recognise 400 Discogs styles) turns every ~2 s of audio
into an **embedding** (1280 numbers describing the sound); a small classifier per mood (the "head") turns each embedding into
two probabilities (mood / not mood). As in the lecture: these are moods **perceived** by annotators, not emotions felt by listeners.

The models are published by the MTG (Universitat Pompeu Fabra) under CC BY-NC-SA 4.0 (non-commercial use): https://essentia.upf.edu/models.html

### 5.2 · Installation
Essentia's Python packages exist **only for Linux and macOS** (no Windows packages, not on conda-forge) – but there,
no compiling is needed. On Windows we therefore use **WSL2**: a real Ubuntu Linux that runs inside Windows and can read your Windows files.

| Your system | Way |
|---|---|
| **Windows 10 (version 2004 or newer) / Windows 11** | **A** WSL2 → **B** new environment → **C** JupyterLab |
| macOS Apple Silicon with macOS ≥ 15, Intel Mac with macOS ≥ 14, Linux (x86-64) | **B** new environment → VSCodium kernel |
| older macOS | **B** with the versions from the table at the end of 5.2 |

Disk space: about 3 GB (Ubuntu, Miniforge, environment, models).

#### A · Windows: set up WSL2 (once)
1. **Check virtualisation:** Task Manager → *Performance* → *CPU* → "Virtualization: **Enabled**".
   If it says *Disabled*: enable *Intel VT-x* / *AMD-V (SVM)* in the BIOS/UEFI settings of your computer first.
2. Open **PowerShell as administrator** (Start → type "PowerShell" → right click → *Run as administrator*; only this step needs admin rights):
   ```powershell
   wsl --install -d Ubuntu
   ```
   Restart Windows when asked.
3. After the restart the **Ubuntu** window opens by itself (otherwise: Start → "Ubuntu").
   Choose a Linux user name and a password (the password is invisible while you type – that's normal).
4. Check in PowerShell (normal, no admin): `wsl -l -v` → the line *Ubuntu* must show **VERSION 2**.
   If it shows 1: `wsl --set-version Ubuntu 2`.
5. In the **Ubuntu** window, install Miniforge for Linux (a separate conda, independent of your Windows Miniforge):
   ```bash
   cd ~
   curl -L -O https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
   bash Miniforge3-Linux-x86_64.sh -b -p "$HOME/miniforge3"    # -b = no questions, install to ~/miniforge3
   "$HOME/miniforge3/bin/conda" init bash                       # activate conda in every new Ubuntu window
   exec bash                                                    # reload the shell
   conda --version                                              # must print a version number
   ```
6. **Your Windows files** are under `/mnt/c/…` in Ubuntu. Convert your project path with `wslpath` (single quotes!):
   ```bash
   cd "$(wslpath 'C:\Users\<you>\<path-to>\ml-tutorium')"
   ls                                                           # must show data/, output/, the notebooks …
   ```

Common problems:
- `WslRegisterDistribution failed with error: 0x80370102` → virtualisation is disabled (step 1).
- `wsl --install` only prints the help text → WSL is already installed: `wsl --update`, then `wsl --install -d Ubuntu`.
- Writing to `output/` fails ("Permission denied") → Windows' *Controlled folder access* blocks it: move the project folder out of *Documents*.
- WSL uses a lot of RAM after work → `wsl --shutdown` in PowerShell.

#### B · New environment (all systems; on Windows: in the Ubuntu window)
```bash
conda create -n mirml-essentia python=3.12 numpy pandas matplotlib ipykernel jupyterlab   # new env, from conda-forge
conda activate mirml-essentia                                                           # never install into base
python -m pip install essentia-tensorflow==2.1b6.dev1389                                # not on conda-forge → pip, in the active env
python -c "import essentia.standard as es; print('ok')"
```
- **Why this version:** the newest release (`2.1b6.dev1438`, May 2026) only has packages for Python 3.14;
  `2.1b6.dev1389` (July 2025) has them for Python 3.9–3.13 → Python 3.12, like `mirml`.
- Install **either** `essentia` **or** `essentia-tensorflow`, never both. Only `essentia-tensorflow` runs the models.
- Messages like `Could not load dynamic library 'libcudart.so…'` only mean "no GPU" – the models run on the CPU.
  `MusicExtractorSVM: no classifier models were configured` is harmless too.
- Write an `environment.yml` for this env as for `mirml` (`essentia-tensorflow` under `- pip:`).

**Older macOS** (no package for your system with the command above):

| Mac | Replace in **B** |
|---|---|
| Apple Silicon, macOS 11–14 | `python=3.12` and `python -m pip install essentia-tensorflow==2.1b6.dev1177 "numpy<2"` |
| Intel, macOS older than 14 | `python=3.11` and `python -m pip install essentia-tensorflow==2.1b6.dev1110 "numpy<2"` |

#### C · Open the notebook in the new environment
- **Windows (WSL):** in the Ubuntu window, in your project folder (step A6), with `mirml-essentia` active:
  ```bash
  jupyter lab --no-browser
  ```
  Copy the printed link `http://localhost:8888/lab?token=…` into your Windows browser and open this notebook.
  Stop JupyterLab with Ctrl+C in the Ubuntu window.
- **macOS / Linux:** VSCodium → *Select Kernel* → `mirml-essentia`.

Then start **here** (not with *Run All*: the cells above need `mirml`).

In [ ]:
# ── Block 5 runs in the kernel "mirml-essentia" ──────────────────────────
import json
import sys
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import essentia
import essentia.standard as es

essentia.log.infoActive = False                       # fewer log messages
print("Python", sys.version.split()[0], "| essentia", essentia.__version__, "|", sys.executable)

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────
CORPUS_DIR = Path("data") / "your-genre"   # ← name of your corpus folder
YEARS = list(range(2013, 2024))            # ← the years YOUR corpus covers, e.g. list(range(2020, 2024))
# ─────────────────────────────────────────────────────────────────────────
SESSION = "T05"
OUTPUT_DIR = Path("output") / SESSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = Path("models") / "essentia"               # downloaded once, reused in later sessions
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MOODS = ["mood_happy", "mood_sad", "mood_relaxed", "mood_aggressive", "mood_party", "mood_acoustic", "mood_electronic"]
HEAD_URL = "https://essentia.upf.edu/models/classification-heads/{m}/{m}-discogs-effnet-1.json"
SEED = 42
N_DEMO = 5

assert CORPUS_DIR.is_dir(), f"Not found: {CORPUS_DIR.resolve()} → check CORPUS_DIR"
PREVIEW_DIR = CORPUS_DIR / "previews"                 # one MP3 per track: previews/<beatport_track_id>.mp3 (built in T02)
assert PREVIEW_DIR.is_dir(), f"Not found: {PREVIEW_DIR.resolve()}"
charts = pd.read_csv(next(CORPUS_DIR.glob("*.csv")))  # chart table: tells us which track belongs to which year

def top100_files(year):
    # Preview files of the Top 100 of one year: IDs from the CSV, kept only if the MP3 exists in previews/
    top = charts[(charts["year"] == year) & (charts["rank"] <= 100)]
    ids = top["beatport_track_id"].dropna().astype(int).unique()
    return [PREVIEW_DIR / f"{i}.mp3" for i in ids if (PREVIEW_DIR / f"{i}.mp3").exists()]

print({year: len(top100_files(year)) for year in YEARS})   # previews per year

**Download the models.** Each model comes as weights (`.pb`) + metadata (`.json`). We read the metadata first:
it contains the download link, the **order of the classes** (differs between models: `["happy", "non_happy"]` but
`["non_relaxed", "relaxed"]`!) and which embedding model it needs.

In [ ]:
def fetch(url, folder=MODEL_DIR):
    # Download once into folder; afterwards just return the local path
    path = folder / url.rsplit("/", 1)[-1]
    if not path.exists():
        print("downloading", path.name)
        urllib.request.urlretrieve(url, path)
    return path

heads = {}
for m in MOODS:
    meta = json.loads(fetch(HEAD_URL.format(m=m)).read_text())
    assert m.removeprefix("mood_") in meta["classes"], f"unexpected class names for {m}: {meta['classes']}"
    heads[m] = {"pb": fetch(meta["link"]), "classes": meta["classes"]}
embedding_pb = fetch(meta["inference"]["embedding_model"]["link"])   # all seven heads share one embedding model

pd.DataFrame({m: h["classes"] for m, h in heads.items()}, index=["class 0", "class 1"]).T

In [ ]:
# Load the models ONCE (slow) – then reuse them for every track (fast)
embedding_model = es.TensorflowPredictEffnetDiscogs(graphFilename=str(embedding_pb), output="PartitionedCall:1")
classifiers = {m: es.TensorflowPredict2D(graphFilename=str(h["pb"]), output="model/Softmax") for m, h in heads.items()}

def track_moods(mp3_path):
    # One preview → dict with 7 confidence scores (0…1), one per mood.
    # 1) load the audio as the models expect it: mono, 16 kHz
    audio = es.MonoLoader(filename=str(mp3_path), sampleRate=16000, resampleQuality=4)()
    # 2) embeddings: one row of 1280 numbers per ~2 s patch of the track → (n_patches, 1280)
    emb = embedding_model(audio)
    scores = {}
    for m, clf in classifiers.items():
        # 3) classifier head: for every patch two probabilities that sum to 1 → (n_patches, 2)
        probs = clf(emb)
        # 4) pick the column of the POSITIVE class ("happy", not "non_happy") – its position differs between models,
        #    so we look it up in the model's metadata – and average over all patches (← choice)
        mood = m.removeprefix("mood_")                # "mood_happy" → "happy"
        scores[mood] = float(probs[:, heads[m]["classes"].index(mood)].mean())
    return scores

mp3s = top100_files(YEARS[-1])                        # Top 100 previews of your latest year
TRACK = mp3s[np.random.default_rng(SEED).integers(len(mp3s))]
scores = track_moods(TRACK)

plt.figure(figsize=(6, 2.5))
plt.bar(scores.keys(), scores.values())
plt.ylim(0, 1)
plt.ylabel("confidence")
plt.title(f"Essentia mood scores – {TRACK.stem}")
plt.show()

**Reading the scores:**
- Seven **independent** binary models → the scores don't add up to 1; "happy" and "sad" can both be low.
- 0.3 means "the model leans towards *not* happy", not "30 % happy".
- The models were trained on general (mostly non-EDM) music: `electronic` will be close to 1 and `acoustic` close to 0 for almost every track – little information in an EDM corpus.
- Averaging over patches is a choice (median, share of patches > 0.5, …).

**Several tracks with cache** (same idea as `enrich()` in block 1):

In [ ]:
YEAR = YEARS[-1]                                      # demo: one year, N_DEMO tracks → later: loop over YEARS
files = top100_files(YEAR)[:N_DEMO]
cache = OUTPUT_DIR / "essentia_moods.csv"
done = set(pd.read_csv(cache)["beatport_track_id"].astype(str)) if cache.exists() else set()

for f in files:
    if f.stem in done:                                # f.stem = file name without ".mp3" = beatport_track_id
        continue
    row = {"year": YEAR, "beatport_track_id": f.stem, **track_moods(f)}
    pd.DataFrame([row]).to_csv(cache, mode="a", header=not cache.exists(), index=False)   # append at once

moods = pd.read_csv(cache)
moods

✎ **Your turn:** run it for the Top 100 of every year in `YEARS` (a few seconds per track on a CPU – let it run, the cache makes it resumable), then aggregate per year,
visualise the development of mood and continue with the SSM & novelty from block 4 – over years.